# Chapter 3: Programming Exercises
### PyTorch Internals and Execution Performance

Seven hands-on exercises, one per major section of the chapter. Every one is about *measuring* something, because that is the chapter's whole argument: don't guess, profile.

**Environment.** Runs on Google Colab free tier. A few exercises (3.3 for memory, 3.6 and 3.7 for the full effect) want a GPU. Set Runtime > Change runtime type > T4 GPU. The rest run on CPU. `torch` and `numpy` are preinstalled on Colab; nothing else to install.

**How to use.** Each exercise is a markdown brief followed by a starter cell with `TODO`s. Fill them in and run. Solutions are in the companion solutions notebook.


## Exercise 3.1: The Python tax

**Section 3.1.** You read that pure Python is 10-100x slower than compiled code for numeric work, and that the gap is execution path, not arithmetic. Prove it on your own machine.

**Task.** Sum one million floats four ways: a pure Python `for` loop, the built-in `sum()`, `numpy.sum()`, and `torch.sum()`. Time each and print the slowdown relative to torch. Then disassemble a one-line expression with `dis` and count how many bytecode instructions one `+` becomes.

**Measure.** Wall-clock time per approach, and the ratio to `torch.sum()`.

**You're done when** the pure-Python loop is two to three orders of magnitude slower than torch, and you can point to the `BINARY_ADD` in the disassembly.

In [ ]:
import time, dis
import numpy as np
import torch

N = 1_000_000
py_list = list(range(N))
np_arr  = np.arange(N, dtype=np.float64)
tt      = torch.arange(N, dtype=torch.float64)

def py_loop_sum(xs):
    # TODO: sum xs with a plain Python for-loop
    raise NotImplementedError

def timeit(fn, repeat=3):
    best = float('inf')
    for _ in range(repeat):
        t0 = time.perf_counter(); fn(); best = min(best, time.perf_counter() - t0)
    return best

# TODO: fill the four approaches, then print a table of time and ratio-to-torch
results = {
    'python loop': None,   # timeit(lambda: py_loop_sum(py_list))
    'builtin sum': None,   # timeit(lambda: sum(py_list))
    'numpy .sum' : None,   # timeit(lambda: np_arr.sum())
    'torch .sum' : None,   # timeit(lambda: tt.sum())
}

# TODO: disassemble this and find BINARY_ADD
def expr(a, b, c):
    return a + b * c
# dis.dis(expr)


## Exercise 3.2: Where launch overhead dominates

**Section 3.2.** A forward pass made of many tiny ops is bound by per-op dispatch, not compute, until the batch grows large enough to hide it. Watch the break-even happen.

**Task.** Build a function that applies 200 small elementwise ops to a tensor. Time one call at batch sizes 1, 8, 64, 512, 4096. Use warmup and a device sync so you measure real execution, not the launch queue.

**Measure.** ms per call and ms per sample at each batch size.

**You're done when** ms-per-call is nearly flat at small batch (the 200 launches dominate) and ms-per-sample falls sharply as batching amortizes them.

In [ ]:
import time, torch
dev = 'cuda' if torch.cuda.is_available() else 'cpu'
print('device:', dev)

def many_small_ops(x, n_ops=200):
    for _ in range(n_ops):
        x = x * 1.0001 + 0.0001
    return x

def bench(batch, n_ops=200, iters=50, warmup=10):
    x = torch.randn(batch, 256, device=dev)
    # TODO: warm up, then time `iters` calls with a device sync on both sides
    # return milliseconds per call
    raise NotImplementedError

# for b in [1, 8, 64, 512, 4096]:
#     ms = bench(b); print(b, ms)


## Exercise 3.3: What autograd costs at inference

**Section 3.3.** At inference, autograd records a graph and saves activations that nobody uses. Measure the latency and memory you get back by turning it off. **Needs a GPU** for the memory numbers (Colab: Runtime > Change runtime type > T4).

**Task.** Run the same forward pass three ways: autograd on, inside `torch.no_grad()`, and inside `torch.inference_mode()`. Record latency and peak GPU memory with `torch.cuda.reset_peak_memory_stats()` and `torch.cuda.max_memory_allocated()`.

**Measure.** Latency and peak GB for each mode.

**You're done when** `no_grad` and `inference_mode` cut peak memory well below the autograd-on case, reproducing the shape of Table 3.5.

In [ ]:
import time, torch
import torch.nn as nn

if not torch.cuda.is_available():
    print('This exercise needs a GPU for the memory numbers.')
else:
    dev = 'cuda'
    model = nn.Sequential(nn.Linear(768, 3072), nn.GELU(),
                          nn.Linear(3072, 768), nn.GELU(),
                          nn.Linear(768, 768)).to(dev).eval()
    x = torch.randn(8, 512, 768, device=dev)

    def run(mode):
        torch.cuda.reset_peak_memory_stats(); torch.cuda.synchronize()
        t0 = time.perf_counter()
        # TODO: run model(x) under the right context for `mode`
        #   'grad'      -> no context
        #   'no_grad'   -> with torch.no_grad():
        #   'inference' -> with torch.inference_mode():
        raise NotImplementedError
        torch.cuda.synchronize()
        return (time.perf_counter()-t0)*1e3, torch.cuda.max_memory_allocated()/1e9

    # for m in ['grad','no_grad','inference']: print(m, run(m))


## Exercise 3.4: Sweeping the DataLoader

**Section 3.4.** `num_workers` and `pin_memory` are the highest-leverage settings you have, and they're free. Measure what they buy on a dataset with real per-sample CPU work.

**Task.** A `Dataset` whose `__getitem__` does a small FFT (standing in for image decode or tokenization) is provided. Sweep `num_workers` over 0, 2, 4, 8 with `pin_memory` on or off and measure samples per second.

**Measure.** Throughput (samples/sec) at each setting.

**You're done when** moving from `num_workers=0` to a few workers gives a multi-fold throughput jump, the same pattern as Table 3.7.

In [ ]:
import time, torch
import numpy as np
from torch.utils.data import Dataset, DataLoader

class SlowDataset(Dataset):
    def __init__(self, n=2000): self.n = n
    def __len__(self): return self.n
    def __getitem__(self, i):
        a = np.random.rand(256, 256)
        a = np.fft.fft2(a).real            # simulated CPU decode/augment
        return torch.from_numpy(a[:64, :64].copy()).float()

def sweep(num_workers, pin, batch=32, max_batches=40):
    # TODO: build a DataLoader with these settings, iterate max_batches,
    # and return samples-per-second. Warm one batch first to spin up workers.
    raise NotImplementedError

# for nw, pin in [(0,False),(2,False),(4,True),(8,True)]:
#     print(nw, pin, sweep(nw, pin))


## Exercise 3.5: Hunt the anti-pattern

**Section 3.5.** The `slow_step()` below works correctly and runs slowly. It hides several of the anti-patterns from the chapter: a Python loop over samples instead of a batched call, a constant kept on the CPU and moved every step, a `.item()` sync on the hot path, and `zero_grad()` without `set_to_none`.

**Task.** Read `slow_step`, name each anti-pattern, then write `fast_step()` that fixes them. Time both.

**Measure.** ms per step, slow versus fast.

**You're done when** `fast_step` is clearly faster and you can list the anti-patterns you removed.

In [ ]:
import time, torch
import torch.nn as nn
dev = 'cuda' if torch.cuda.is_available() else 'cpu'

model = nn.Sequential(nn.Linear(256, 256), nn.ReLU(), nn.Linear(256, 1)).to(dev)
opt = torch.optim.SGD(model.parameters(), lr=1e-3)
batch = [torch.randn(256, device=dev) for _ in range(64)]   # list of samples
scale_cpu = torch.tensor(0.5)                                # lives on CPU

def slow_step():
    outs = []
    for s in batch:                     # anti-pattern: per-sample loop
        outs.append(model(s.unsqueeze(0)))
    out = torch.cat(outs, 0)
    out = out * scale_cpu.to(dev)       # anti-pattern: CPU constant moved each call
    loss = out.pow(2).mean()
    opt.zero_grad()                     # anti-pattern: keeps grad buffers as zeros
    loss.backward(); opt.step()
    return loss.item()                  # anti-pattern: sync every step

def fast_step():
    # TODO: one batched forward, on-device constant, set_to_none=True,
    # and don't sync every step (return the loss tensor, log occasionally)
    raise NotImplementedError

def timeit(fn, iters=100, warmup=10):
    for _ in range(warmup): fn()
    if dev == 'cuda': torch.cuda.synchronize()
    t0 = time.perf_counter()
    for _ in range(iters): fn()
    if dev == 'cuda': torch.cuda.synchronize()
    return (time.perf_counter() - t0) / iters * 1e3

# print('slow', timeit(slow_step)); print('fast', timeit(fast_step))


## Exercise 3.6: Benchmark without lying to yourself

**Section 3.6.** Timing a CUDA kernel with `time.perf_counter()` and no sync measures the launch queue, not the work, and you can be off by a thousand-fold. See the error, then do it right.

**Task.** Time a 4096x4096 matmul the wrong way (no warmup, no sync) and the right way (warmup plus CUDA events, or a synced loop on CPU). Then confirm with `torch.utils.benchmark.Timer`.

**Measure.** The wrong time, the right time, and the gap between them.

**You're done when** the wrong measurement is implausibly small and the CUDA-event number is the real per-call cost.

In [ ]:
import time, torch
dev = 'cuda' if torch.cuda.is_available() else 'cpu'
a = torch.randn(4096, 4096, device=dev)
b = torch.randn(4096, 4096, device=dev)

# WRONG: no warmup, no sync
t0 = time.perf_counter(); c = a @ b
wrong_ms = (time.perf_counter() - t0) * 1e3

# TODO: RIGHT way -- warm up, then time with CUDA events (GPU) or a synced loop (CPU)
right_ms = None

print('wrong', wrong_ms, 'right', right_ms)


## Exercise 3.7: Capstone, optimize the embedding stage

**Section 3.7.** Put the whole checklist to work on the RAG embedding step. A small transformer encoder stands in for BGE-M3 so the notebook runs anywhere; swap in the real model later if you like.

**Task.** The `naive()` function embeds 256 queries one at a time, with autograd on and a blocking transfer per query. Write `optimized()` that uses `torch.inference_mode()`, batches all queries into one forward, and transfers once with `non_blocking=True`. Time both.

**Measure.** Total time and ms-per-query for each, and the speedup.

**You're done when** the optimized path is several times faster per query, the same story as Table 3.9. (On a GPU the gap is largest; on CPU you'll still see batching win.)

In [ ]:
import time, torch
import torch.nn as nn
dev = 'cuda' if torch.cuda.is_available() else 'cpu'

class TinyEncoder(nn.Module):
    def __init__(self, d=384, layers=6, heads=6):
        super().__init__()
        layer = nn.TransformerEncoderLayer(d, heads, dim_feedforward=4*d, batch_first=True)
        self.body = nn.TransformerEncoder(layer, layers)
    def forward(self, x):          # x: [B, T, d]
        return self.body(x).mean(1)

model = TinyEncoder().to(dev).eval()
queries = [torch.randn(64, 384) for _ in range(256)]   # 256 queries on CPU

def naive():
    embs = []
    for q in queries:                       # batch=1, autograd on, blocking move
        qd = q.unsqueeze(0).to(dev)
        embs.append(model(qd))
    return torch.cat(embs, 0)

def optimized():
    # TODO: inference_mode, one batched transfer (non_blocking), one batched forward
    raise NotImplementedError

def timeit(fn, iters=5, warmup=2):
    for _ in range(warmup): fn()
    if dev == 'cuda': torch.cuda.synchronize()
    t0 = time.perf_counter()
    for _ in range(iters): fn()
    if dev == 'cuda': torch.cuda.synchronize()
    return (time.perf_counter() - t0) / iters

# n = len(queries)
# ts = timeit(naive); to = timeit(optimized)
# print(ts, to, ts/to)
